# Replication

Reruns the MNIST pass-rate check on the published RBT4DNN generated images. This is an artifact-level replication: it does not retrain LoRAs or regenerate images.


In [ ]:
from pathlib import Path
import csv
import subprocess

REPO = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
DATA = Path("data/requirements.csv")
roots = [Path.cwd(), *Path.cwd().parents]

path = next((root / DATA for root in roots if (root / DATA).exists()), None)
if path is None:
    repo = Path("/content/rbt4dnn-seminar")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(repo)], check=True)
    path = repo / DATA

if not path.exists():
    raise FileNotFoundError(f"Could not find {DATA}")

ROOT = path.parents[1]
with path.open(newline="") as f:
    rows = list(csv.DictReader(f))

print(path)

In [1]:
def write_csv(path, fieldnames, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


out_rows = []
for r in rows:
    if r["dataset"] == "mnist" and r["variant"] == "lr" and r["local_replication_pass_rate_n100"]:
        local = float(r["local_replication_pass_rate_n100"])
        paper = float(r["paper_reference_pass_rate"])
        out_rows.append(
            {
                "requirement": r["requirement"],
                "local_pass_rate_n100": f"{local:.3f}",
                "paper_pass_rate": f"{paper:.3f}",
                "delta": f"{local - paper:+.3f}",
            }
        )

out = ROOT / "experiments" / "replication" / "results.csv"
write_csv(out, ["requirement", "local_pass_rate_n100", "paper_pass_rate", "delta"], out_rows)

print("req | local n=100 | paper ref | delta")
for r in out_rows:
    print(
        f"{r['requirement']} | {r['local_pass_rate_n100']} | {r['paper_pass_rate']} | {r['delta']}"
    )
print(out)

req | local n=100 | paper ref | delta
M1 | 1.000 | 0.999 | +0.001
M2 | 0.990 | 0.977 | +0.013
M3 | 0.730 | 0.724 | +0.006
M4 | 0.970 | 0.982 | -0.012
M5 | 1.000 | 0.994 | +0.006
M6 | 0.980 | 0.976 | +0.004
M7 | 0.980 | 0.981 | -0.001


Takeaway: the local MNIST pass rates closely match the paper-release values, so the copied generated-image artifact and evaluation setup are usable for the extension work.